# Corrected model with per-group thresholds

Create a qualification reference label from one pooled `q` cutoff, retrain the baseline RandomForest scoring model, and select thresholds that meet both the equal-opportunity and grant-budget constraints. `predictions.csv` is written only when a feasible pair exists.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
TAG="hari_feature_test"
historique = pd.read_csv(f'data/q_historique_{TAG}.csv')
evaluation = pd.read_csv(f'data/q_evaluation_{TAG}.csv')

# Same grouping rule as baseline_model_en.ipynb.
ELOIGNEES = ['Bas-Saint-Laurent', 'Cote-Nord', 'Gaspesie-Iles-de-la-Madeleine']
def groupe_region(df):
    return np.where(df['region_administrative'].isin(ELOIGNEES), 'Eloignee', 'Centre')

for df in (historique, evaluation):
    if 'groupe' not in df:
        df['groupe'] = groupe_region(df)

# One cutoff at the 60th percentile of q pooled over both files and groups.
q_cutoff = pd.concat([historique['q'], evaluation['q']], ignore_index=True).quantile(0.60)
historique['qualifie'] = (historique['q'] >= q_cutoff).astype(int)
evaluation['qualifie'] = (evaluation['q'] >= q_cutoff).astype(int)
print(f'Pooled q cutoff for top 40%: {q_cutoff:.8f}')
print('Historical qualifie rate by groupe:')
print(historique.groupby('groupe')['qualifie'].mean().to_string(float_format=lambda x: f'{x:.4f}'))
print('Evaluation qualifie rate by groupe:')
print(evaluation.groupby('groupe')['qualifie'].mean().to_string(float_format=lambda x: f'{x:.4f}'))

Pooled q cutoff for top 40%: 0.67890112
Historical qualifie rate by groupe:
groupe
Centre     0.4115
Eloignee   0.3728
Evaluation qualifie rate by groupe:
groupe
Centre     0.4195
Eloignee   0.3962


## Retrain baseline scoring model

In [2]:
CATEGORIELLES = ['programme_etudes', 'region_administrative', 'code_postal_3']
def encoder(df_entrainement, df_cible):
    X = pd.get_dummies(
        df_entrainement.drop(columns=['id_candidat', 'decision_octroi', 'groupe', 'q', 'qualifie'], errors='ignore'),
        columns=CATEGORIELLES)
    Xc = pd.get_dummies(
        df_cible.drop(columns=['id_candidat', 'decision_octroi', 'groupe', 'q', 'qualifie'], errors='ignore'),
        columns=CATEGORIELLES).reindex(columns=X.columns, fill_value=0)
    return X, Xc

X_hist, X_eval = encoder(historique, evaluation)
modele = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
modele.fit(X_hist, historique['decision_octroi'])
score = modele.predict_proba(X_eval)[:, 1]
evaluation['s'] = score
print(f'Scored {len(evaluation):,} evaluation applicants using predict_proba.')

Scored 4,000 evaluation applicants using predict_proba.


## Search threshold pairs

In [3]:
# Enumerate each distinct selection obtainable via s >= t. Thresholds are
# distinct observed scores plus a value just above the maximum (select nobody).
def threshold_options(frame, group):
    scores = frame.loc[frame['groupe'].eq(group), 's'].to_numpy()
    return np.r_[np.unique(scores), np.nextafter(scores.max(), np.inf)]

centre = evaluation[evaluation['groupe'].eq('Centre')]
eloignee = evaluation[evaluation['groupe'].eq('Eloignee')]
tc_values = threshold_options(evaluation, 'Centre')
te_values = threshold_options(evaluation, 'Eloignee')

def option_stats(frame, thresholds):
    scores = frame['s'].to_numpy()
    qualified = frame['qualifie'].to_numpy().astype(bool)
    selected = scores[None, :] >= thresholds[:, None]
    grants = selected.sum(axis=1)
    q_total = qualified.sum()
    q_selected = (selected & qualified[None, :]).sum(axis=1)
    tpr = q_selected / q_total if q_total else np.zeros(len(thresholds))
    grant_rate = grants / len(frame)
    return grants, q_selected, tpr, grant_rate

cg, cq, ctpr, cgr = option_stats(centre, tc_values)
eg, eq, etpr, egr = option_stats(eloignee, te_values)
# Vectorized pair grid: EO gap <= 1 percentage point and overall grant rate 38%-42%.
gap = np.abs(ctpr[:, None] - etpr[None, :])
overall_rate = (cg[:, None] + eg[None, :]) / len(evaluation)
qual_selected = cq[:, None] + eq[None, :]
feasible = (gap <= 0.01 + 1e-12) & (overall_rate >= 0.38) & (overall_rate <= 0.42)

if feasible.any():
    # Maximize qualified applicants selected; deterministic tie-break by smaller EO gap,
    # then overall grant rate nearest 40%, then lower thresholds.
    candidates = np.argwhere(feasible)
    order = sorted(candidates.tolist(), key=lambda ij: (
        -int(qual_selected[ij[0], ij[1]]), float(gap[ij[0], ij[1]]),
        abs(float(overall_rate[ij[0], ij[1]]) - 0.40),
        float(tc_values[ij[0]]), float(te_values[ij[1]])))
    i, j = order[0]
    chosen_c, chosen_e = tc_values[i], te_values[j]
    pred = np.where(evaluation['groupe'].eq('Centre'), score >= chosen_c, score >= chosen_e).astype(int)
    evaluation['prediction'] = pred
    evaluation['qualifie_selected'] = evaluation['qualifie'] & evaluation['prediction']
    print(f'Chosen thresholds: Centre={chosen_c:.8f}, Eloignee={chosen_e:.8f}')
    print(f'TPR Centre={ctpr[i]:.4f}; Eloignee={etpr[j]:.4f}; EO gap={gap[i,j]:.4f}')
    print(f'Grant rate Centre={cgr[i]:.4f}; Eloignee={egr[j]:.4f}; overall={overall_rate[i,j]:.4f}')
    print(f'Qualified applicants selected: {qual_selected[i,j]:,}')
else:
    # Report closest budget-feasible pairs by EO gap (if any); otherwise closest
    # overall-budget combinations across the grid. No budget relaxation.
    budget_ok = (overall_rate >= 0.38) & (overall_rate <= 0.42)
    if budget_ok.any():
        pool = np.argwhere(budget_ok)
        label = 'Closest combinations within budget, ranked by EO gap:'
    else:
        pool = np.argwhere(np.ones_like(overall_rate, dtype=bool))
        label = 'No pair meets budget; closest combinations by budget distance, then EO gap:'
    ranked = sorted(pool.tolist(), key=lambda ij: (
        (float(gap[ij[0],ij[1]]) if budget_ok.any() else max(0.38-float(overall_rate[ij[0],ij[1]]), float(overall_rate[ij[0],ij[1]])-0.42, 0)),
        float(gap[ij[0],ij[1]]), abs(float(overall_rate[ij[0],ij[1]])-0.40)))[:10]
    print('STOP: no threshold pair satisfies both EO gap <= 0.01 and grant rate 0.38-0.42.')
    print(label)
    for ii,jj in ranked:
        print(f'Centre t={tc_values[ii]:.8f}; Eloignee t={te_values[jj]:.8f}; TPR=({ctpr[ii]:.4f},{etpr[jj]:.4f}); EO gap={gap[ii,jj]:.4f}; overall grant rate={overall_rate[ii,jj]:.4f}')

Chosen thresholds: Centre=0.61342383, Eloignee=0.18771005
TPR Centre=0.9307; Eloignee=0.9318; EO gap=0.0011
Grant rate Centre=0.4279; Eloignee=0.4085; overall=0.4200
Qualified applicants selected: 1,527


## Write and verify predictions

In [4]:
if feasible.any():
    predictions = pd.DataFrame({
        'id_candidat': evaluation['id_candidat'],
        'decision_octroi': evaluation['prediction'].astype(int),
    })
    predictions.to_csv('predictions.csv', index=False)
    readback = pd.read_csv('predictions.csv')
    overall = readback['decision_octroi'].mean()
    ok = 0.38 <= overall <= 0.42
    print(f'Rows: {len(readback)}')
    print(f'Columns: {readback.columns.tolist()}')
    print(f'Unique decision_octroi values: {sorted(readback["decision_octroi"].unique().tolist())}')
    print(f'Overall grant rate: {overall:.4f} - {"OK" if ok else "OUT OF BUDGET"}')
    verification = evaluation.assign(decision_octroi=readback['decision_octroi'].to_numpy())
    grant_by_group = verification.groupby('groupe')['decision_octroi'].mean()
    tpr_by_group = verification.groupby('groupe').apply(
        lambda g: g.loc[g['qualifie'].eq(1), 'decision_octroi'].mean(), include_groups=False)
    print('Grant rate by groupe:')
    print(grant_by_group.to_string(float_format=lambda x: f'{x:.4f}'))
    print('TPR by groupe against qualifie:')
    print(tpr_by_group.to_string(float_format=lambda x: f'{x:.4f}'))
    print(f'EO gap: {abs(tpr_by_group["Centre"] - tpr_by_group["Eloignee"]):.4f}')
else:
    print('Final verification skipped: no predictions.csv written because no feasible threshold pair was found.')

Rows: 4000
Columns: ['id_candidat', 'decision_octroi']
Unique decision_octroi values: [0, 1]
Overall grant rate: 0.4200 - OK
Grant rate by groupe:
groupe
Centre     0.4279
Eloignee   0.4085
TPR by groupe against qualifie:
groupe
Centre     0.9307
Eloignee   0.9318
EO gap: 0.0011
